# ETL: Entregable 1

In [2]:
from dotenv import dotenv_values
from sqlalchemy import create_engine
from sqlalchemy import text
import datetime
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import redshift_connector

In [3]:
def check_if_valid_data(df: pd.DataFrame) -> bool:
    # Check if dataframe is empty
    if df.empty:
        print("No songs downloaded. Finishing execution")
        return False
    
    # Primary Key check
    if not df["played_at"].is_unique:
        raise Exception("A value from played_at is not unique")
    
    # Check for nulls
    if df.isnull().values.any():
        raise Exception("A value in df is null")
        
    return True

In [4]:
config = dotenv_values(".env")

In [5]:
# Spotifg settings
CLIENT_ID = config["CLIENT_ID"]
CLIENT_SECRET = config["CLIENT_SECRET"]
SPOTIPY_REDIRECT_URI = config["SPOTIPY_REDIRECT_URI"]
SCOPE = config["SCOPE"]

In [96]:
# DB settings
TABLENAME = config["TABLENAME"]
DB_USERNAME = config["DB_USERNAME"]
DB_PASSWORD = config["DB_PASSWORD"]
DB_HOST = config["DB_HOST"]
DB_PORT = config["DB_PORT"]
DB_NAME = config["DB_NAME"]
DB_SCHEMA = config["DB_SCHEMA"]

In [6]:
# Connect to Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=CLIENT_ID,
                                               client_secret=CLIENT_SECRET,
                                               redirect_uri=SPOTIPY_REDIRECT_URI,
                                               scope=SCOPE))

# Convert a unix timestamp in milliseconds
today = datetime.datetime.now()
yesterday = today - datetime.timedelta(days=1)
yesterday = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
yesterday_unix_timestamp = int(yesterday.timestamp()) * 1000
limit = 50    

# Retorna un diccionario con las canciones escuchadas
raw_data = sp.current_user_recently_played(limit=limit, after=yesterday_unix_timestamp)

In [6]:
raw_data.keys()

dict_keys(['items', 'next', 'cursors', 'limit', 'href'])

In [7]:
song_names = []
artist_names = []
played_at_list = []
timestamps = []

In [8]:
for song in raw_data["items"]:
    song_names.append(song["track"]["name"])
    artist_names.append(song["track"]["artists"][0]["name"])
    played_at_list.append(song["played_at"]),
    timestamps.append(song["played_at"][0:10])

In [9]:
song_dict = {
    "song_name" : song_names,
    "artist_name": artist_names,
    "played_at" : played_at_list,
    "timestamp" : timestamps
}

In [10]:
song_df = pd.DataFrame(song_dict, columns = ["song_name",
                                             "artist_name",
                                             "played_at",
                                             "timestamp"])

In [11]:
song_df.head(5)

,song_name,artist_name,played_at,timestamp
0,Under Pressure,Queen,2023-05-27T21:46:13.451Z,2023-05-27
1,I Believe in Miracles,Ramones,2023-05-27T21:42:08.797Z,2023-05-27
2,Wall of Glass,Liam Gallagher,2023-05-27T21:38:49.763Z,2023-05-27
3,Even Better Than The Real Thing,U2,2023-05-27T21:35:06.045Z,2023-05-27
4,The Hindu Times,Oasis,2023-05-27T21:31:24.559Z,2023-05-27


In [12]:
# eliminar fechas diferentes a las que queremos
clean_song_df = song_df[pd.to_datetime(song_df["played_at"]).dt.date == yesterday.date()]

In [13]:
clean_song_df

,song_name,artist_name,played_at,timestamp
18,I Wanna Be Sedated,Ramones,2023-05-26T21:35:05.558Z,2023-05-26
19,Someday,The Strokes,2023-05-26T21:32:35.482Z,2023-05-26
20,D is for Dangerous,Arctic Monkeys,2023-05-26T21:29:30.774Z,2023-05-26
21,Sound Of A Gun,Audioslave,2023-05-26T21:27:12.314Z,2023-05-26
22,Drain You,Nirvana,2023-05-26T21:22:51.902Z,2023-05-26
23,Stop Crying Your Heart Out,Oasis,2023-05-26T21:19:07.646Z,2023-05-26
24,Walk,Foo Fighters,2023-05-26T21:14:04.151Z,2023-05-26
25,Creep,Radiohead,2023-05-26T21:09:47.858Z,2023-05-26
26,What's My Age Again?,blink-182,2023-05-26T21:05:48.909Z,2023-05-26
27,On Melancholy Hill,Gorillaz,2023-05-26T21:03:20.228Z,2023-05-26


In [14]:
# Data validation
if check_if_valid_data(clean_song_df):
    print("Data valid, proceed to Load stage...")

Data valid, proceed to Load stage...


# Conexión a PostgreSQL en local

In [88]:
# # Load

# rows_imported = 0

# conn_string = f"postgresql+psycopg2://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
# engine = create_engine(conn_string)
# conn = engine.connect()

# sql_query = text(
# """
# CREATE TABLE IF NOT EXISTS coder.my_tracks_history(
#     id_my_tracks_history SERIAL NOT NULL,
#     song_name VARCHAR(200) NOT NULL,
#     artist_name VARCHAR(200) NOT NULL,
#     played_at TIMESTAMP NOT NULL,
#     timestamp DATE NOT NULL,

#     CONSTRAINT pk_my_tracks_history PRIMARY KEY (played_at)
# );
# """)

# conn.execute(sql_query)
# conn.commit()

# print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {TABLENAME}")

# clean_song_df.to_sql(TABLENAME,
#                con=engine,
#                schema=DB_SCHEMA,
#                index=False,
#                if_exists="append")

# rows_imported += clean_song_df.shape[0]
# print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {TABLENAME}")
# print(f"Data imported successful")

# conn.close()

Importing rows 0 to 5... for table my_tracks_history
Importing rows 5 to 5... for table my_tracks_history
Data imported successful


# Conexión a AWS Redshift

In [15]:
# Load

# AWS Redshift settings
REDSHIFT_HOST = config["AWS_HOST"]
REDSHIFT_PORT = int(config["AWS_PORT"])
REDSHIFT_DATABASE = config["AWS_DB_NAME"]
REDSHIFT_SCHEMA = config["AWS_DB_SCHEMA"]
REDSHIFT_USERNAME = config["AWS_USER"]
REDSHIFT_PASS = config["AWS_PASSWORD"]

In [16]:
#Connect to the cluster
conn = redshift_connector.connect(
    host=REDSHIFT_HOST,
    port=REDSHIFT_PORT,
    database=REDSHIFT_DATABASE,
    user=REDSHIFT_USERNAME,
    password=REDSHIFT_PASS
)

# Create a Cursor object
cursor = conn.cursor()


In [23]:
conn_string = f"redshift+psycopg2://{REDSHIFT_USERNAME}:{REDSHIFT_PASS}@{REDSHIFT_HOST}:{REDSHIFT_PORT}/{REDSHIFT_DATABASE}"
engine = create_engine(conn_string)
conn_ = engine.connect()